In [ ]:
# DINOv3-Guided YOLOv12m Pretraining (Part B, SSL)

**Student:** randomly initialized YOLOv12m backbone from `yolo12m.yaml`
**Teacher:** frozen released DINOv3 ViT-B/16
**Data:** AgroPest-12-PartB-Safe SSL pool only (9,180 unlabelled images)
**Colab validation:** one T4 GPU, one full SSL-pool epoch

This is standalone PyTorch distillation. It does not use Lightly, PyTorch Lightning, or an SSL library. The student backbone executes the original Ultralytics YAML graph using each module's `f`, `i`, and `save` routing metadata. Only the student encoder transfers downstream.

In [1]:
# CONFIG: Colab one-T4 validation. Kaggle conversion changes paths, epoch count, and DDP only.
from dataclasses import asdict, dataclass
from pathlib import Path

@dataclass
class CFG:
    # Image-only input. This notebook must never inspect labels, valid, or test paths.
    ssl_pool: str = "/content/partb_safe/ssl_pool/images"
    output_dir: str = "/content/partB_04a_dinov3_yolo12m"
    teacher_ckpt: str = ""

    seed: int = 445
    model_yaml: str = "yolo12m.yaml"  # random initialization; never a .pt checkpoint.
    image_size: int = 224              # compatible with the ViT-B/16 14x14 patch grid.
    teacher_embed_dim: int = 768       # DINOv3 ViT-B/16 feature dimension.
    teacher_patch_size: int = 16

    epochs: int = 1                    # smoke validation; Kaggle experiment uses 30.
    batch_size: int = 8                # per GPU, conservative for one 16-GB T4 with ViT teacher.
    grad_accum: int = 2                # effective batch 16; does not change teacher targets.
    workers: int = 2
    amp: bool = True
    max_train_images: int | None = None  # None means all 9,180 SSL-pool images.

    learning_rate: float = 3e-4
    min_learning_rate: float = 3e-6
    warmup_epochs: int = 1
    weight_decay: float = 1e-4
    gradient_clip_norm: float = 5.0

    global_loss_weight: float = 1.0
    dense_loss_weight: float = 1.0
    diagnostic_images: int = 96
    retrieval_neighbors: int = 5

cfg = CFG()
OUTPUT_DIR = Path(cfg.output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(asdict(cfg))

{'ssl_pool': '/content/partb_safe/ssl_pool/images', 'output_dir': '/content/partB_04a_dinov3_yolo12m', 'teacher_ckpt': '', 'seed': 445, 'model_yaml': 'yolo12m.yaml', 'image_size': 224, 'teacher_embed_dim': 768, 'teacher_patch_size': 16, 'epochs': 1, 'batch_size': 8, 'grad_accum': 2, 'workers': 2, 'amp': True, 'max_train_images': None, 'learning_rate': 0.0003, 'min_learning_rate': 3e-06, 'warmup_epochs': 1, 'weight_decay': 0.0001, 'gradient_clip_norm': 5.0, 'global_loss_weight': 1.0, 'dense_loss_weight': 1.0, 'diagnostic_images': 96, 'retrieval_neighbors': 5}


In [9]:
# Keep Colab's CUDA-matched PyTorch. Install only runtime utilities required by DINOv3 and the notebook.
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "ultralytics>=8.4.96", "scikit-learn", "seaborn", "kagglehub", "torchmetrics",
])

import torch, ultralytics
assert torch.cuda.is_available(), "Enable a T4 GPU runtime before continuing."
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))
torch.set_float32_matmul_precision("high")

Python: 3.12.13
PyTorch: 2.11.0+cu128
Ultralytics: 8.4.116
GPU: Tesla T4


In [3]:
# Download the public Part B dataset, then retain only the image-only SSL-pool directory.
import shutil
import kagglehub

ssl_pool = Path(cfg.ssl_pool)
if not ssl_pool.is_dir() or len(list(ssl_pool.glob("*"))) != 9180:
    source_root = Path(kagglehub.dataset_download("sadikshahriar/agropest-12-partb-safe"))
    dataset_root = source_root / "AgroPest-12-PartB-Safe"
    assert dataset_root.is_dir(), f"Unexpected Kaggle dataset layout: {source_root}"
    source_ssl = dataset_root / "ssl_pool" / "images"
    assert source_ssl.is_dir(), f"Missing SSL pool: {source_ssl}"
    if ssl_pool.exists():
        shutil.rmtree(ssl_pool)
    ssl_pool.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_ssl, ssl_pool)

ssl_paths = sorted(p for p in ssl_pool.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
assert len(ssl_paths) == 9180, f"Expected 9,180 SSL images, found {len(ssl_paths)}"
assert not any(ssl_pool.rglob("*.txt")), "SSL pool must not contain labels"
print(f"SSL pool validated: {len(ssl_paths):,} image-only files at {ssl_pool}")

100%|██████████| 534M/534M [00:05<00:00, 98.8MB/s]

Extracting files...


SSL pool validated: 9,180 image-only files at /content/partb_safe/ssl_pool/images


In [6]:
# Resolve the public DINOv3 ViT-B/16 checkpoint used by the course reference.
teacher_cache = Path("/root/.cache/kagglehub/datasets/mrifatrashid/dinov3-weigths")
teacher_candidates = sorted(teacher_cache.rglob("dinov3_vitb16_*.pth")) if teacher_cache.exists() else []
if not teacher_candidates:
    teacher_download = Path(kagglehub.dataset_download("mrifatrashid/dinov3-weigths"))
    teacher_candidates = sorted(teacher_download.rglob("dinov3_vitb16_*.pth"))
assert teacher_candidates, "Could not resolve the DINOv3 ViT-B/16 checkpoint"
cfg.teacher_ckpt = str(teacher_candidates[0])
print("Teacher checkpoint:", cfg.teacher_ckpt)
print("Size (MB):", round(Path(cfg.teacher_ckpt).stat().st_size / 1e6, 1))

Teacher checkpoint: /root/.cache/kagglehub/datasets/mrifatrashid/dinov3-weigths/versions/2/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth
Size (MB): 342.9


In [5]:
# Check whether the teacher archive extraction has completed.
cache = Path("/root/.cache/kagglehub/datasets/mrifatrashid/dinov3-weigths")
print("cache exists:", cache.exists())
if cache.exists():
    candidates = list(cache.rglob("dinov3_vitb16_*.pth"))
    print("teacher candidates:", [(str(p), round(p.stat().st_size / 1e6, 1)) for p in candidates])

cache exists: True
teacher candidates: [('/root/.cache/kagglehub/datasets/mrifatrashid/dinov3-weigths/versions/2/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth', 342.9)]


In [7]:
import csv
import json
import math
import random
import time
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from ultralytics import YOLO

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
DEVICE = torch.device("cuda")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

def cosine_regression(prediction, target):
    return (1.0 - F.cosine_similarity(prediction, target, dim=1)).mean()

class SSLImageDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.paths = list(image_paths)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))

def build_train_transform(image_size):
    return transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.5, 1.0), ratio=(0.75, 4 / 3)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(0.35, 0.35, 0.20, 0.06)], p=0.8),
        transforms.RandomGrayscale(p=0.1),
        transforms.RandomApply([transforms.GaussianBlur(23, sigma=(0.1, 2.0))], p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def build_eval_transform(image_size):
    return transforms.Compose([
        transforms.Resize(int(image_size * 256 / 224)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class YOLOGraphBackbone(nn.Module):
    """Executes the native YAML backbone graph, preserving Ultralytics routing."""
    def __init__(self, detector_model):
        super().__init__()
        self.model = detector_model
        self.backbone_end = len(detector_model.yaml["backbone"])
        self.save = set(detector_model.save)
        assert self.backbone_end > 0

    def forward_feature_map(self, x):
        saved = []
        for module in self.model.model:
            if module.i >= self.backbone_end:
                break
            source = module.f
            if source != -1:
                x = saved[source] if isinstance(source, int) else [x if j == -1 else saved[j] for j in source]
            x = module(x)
            saved.append(x if module.i in self.save else None)
        if not isinstance(x, torch.Tensor):
            raise TypeError("YOLO backbone must end in a tensor feature map")
        return x

    def forward(self, x):
        return F.adaptive_avg_pool2d(self.forward_feature_map(x), 1).flatten(1)

    def backbone_state_dict(self):
        """Return only backbone tensors with standard Ultralytics model.N key names."""
        source = self.model.state_dict()
        selected = {}
        for index in range(self.backbone_end):
            prefix = f"model.{index}."
            selected.update({key: value.detach().cpu() for key, value in source.items() if key.startswith(prefix)})
        return selected

class FrozenDINOv3Teacher(nn.Module):
    def __init__(self, teacher, embed_dim, patch_grid):
        super().__init__()
        self.teacher = teacher.eval()
        self.embed_dim = embed_dim
        self.patch_grid = patch_grid
        for parameter in self.teacher.parameters():
            parameter.requires_grad_(False)

    @torch.no_grad()
    def forward(self, images):
        features = self.teacher.forward_features(images)
        global_tokens = features["x_norm_clstoken"]
        patch_tokens = features["x_norm_patchtokens"]
        patch_map = patch_tokens.transpose(1, 2).reshape(
            images.shape[0], self.embed_dim, self.patch_grid, self.patch_grid
        )
        return global_tokens, patch_map

class DistillationStudent(nn.Module):
    def __init__(self, encoder, feature_dim, teacher_dim, global_weight, dense_weight):
        super().__init__()
        self.encoder = encoder
        self.global_head = nn.Sequential(
            nn.Linear(feature_dim, feature_dim, bias=False),
            nn.BatchNorm1d(feature_dim),
            nn.GELU(),
            nn.Linear(feature_dim, teacher_dim),
        )
        self.dense_head = nn.Conv2d(feature_dim, teacher_dim, kernel_size=1)
        self.global_weight = global_weight
        self.dense_weight = dense_weight

    def forward(self, images, teacher_global, teacher_dense):
        features = self.encoder.forward_feature_map(images)
        student_global = self.global_head(F.adaptive_avg_pool2d(features, 1).flatten(1))
        student_dense = F.interpolate(
            self.dense_head(features), size=teacher_dense.shape[-2:], mode="bilinear", align_corners=False
        )
        global_loss = cosine_regression(F.normalize(student_global, dim=1), F.normalize(teacher_global, dim=1))
        dense_loss = cosine_regression(F.normalize(student_dense, dim=1), F.normalize(teacher_dense, dim=1))
        total_loss = self.global_weight * global_loss + self.dense_weight * dense_loss
        return total_loss, {"global_loss": global_loss.detach(), "dense_loss": dense_loss.detach()}

def build_student(config):
    detector = YOLO(config.model_yaml).model.to(DEVICE)
    encoder = YOLOGraphBackbone(detector).to(DEVICE)
    with torch.no_grad():
        feature_dim = encoder(torch.zeros(2, 3, config.image_size, config.image_size, device=DEVICE)).shape[1]
    return encoder, int(feature_dim)

def load_teacher(config):
    # DINOv3 is loaded without pretrained=True so this exact public checkpoint is used.
    raw_teacher = torch.hub.load("facebookresearch/dinov3", "dinov3_vitb16", pretrained=False)
    state = torch.load(config.teacher_ckpt, map_location="cpu", weights_only=False)
    if "state_dict" in state:
        state = state["state_dict"]
    raw_teacher.load_state_dict(state, strict=True)
    teacher = FrozenDINOv3Teacher(raw_teacher, config.teacher_embed_dim, config.image_size // config.teacher_patch_size)
    return teacher.to(DEVICE)

def set_lr(optimizer, base_lr, min_lr, update, total_updates, warmup_updates):
    if update < warmup_updates:
        lr = base_lr * (0.1 + 0.9 * (update + 1) / max(1, warmup_updates))
    else:
        progress = (update - warmup_updates) / max(1, total_updates - warmup_updates - 1)
        lr = min_lr + (base_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))
    for group in optimizer.param_groups:
        group["lr"] = lr
    return lr

def export_detector_checkpoint(config, model, feature_dim, history):
    backbone_state = model.encoder.backbone_state_dict()
    torch.save({
        "format": "yolo_backbone_only_v1",
        "model_yaml": config.model_yaml,
        "backbone_state_dict": backbone_state,
        "feature_dim": feature_dim,
        "config": asdict(config),
    }, OUTPUT_DIR / "backbone_only.pt")

    detector = YOLO(config.model_yaml).model
    detector_state = detector.state_dict()
    expected = {k: v for k, v in detector_state.items() if int(k.split(".")[1]) < model.encoder.backbone_end}
    missing = sorted(set(expected) - set(backbone_state))
    mismatched = sorted(k for k in expected if k in backbone_state and expected[k].shape != backbone_state[k].shape)
    unexpected = sorted(set(backbone_state) - set(expected))
    assert not missing and not mismatched and not unexpected, {
        "missing": missing[:5], "mismatched": mismatched[:5], "unexpected": unexpected[:5]
    }
    detector.load_state_dict(backbone_state, strict=False)
    detector.args = {"model": config.model_yaml, "task": "detect"}
    checkpoint = {
        "model": deepcopy(detector).half().cpu(), "ema": None, "updates": 0,
        "optimizer": None, "train_args": detector.args,
        "date": time.strftime("%Y-%m-%dT%H:%M:%S"), "version": ultralytics.__version__,
    }
    detector_path = OUTPUT_DIR / "yolo12m_dinov3_backbone.pt"
    torch.save(checkpoint, detector_path)
    report = {
        "source": "random yolo12m.yaml distilled with frozen DINOv3 ViT-B/16",
        "transferred_component": "YOLOv12m backbone only",
        "excluded_components": ["global_head", "dense_head", "optimizer", "scheduler", "detector_neck", "detector_head"],
        "backbone_tensor_coverage": {"matched": len(expected), "expected": len(expected), "percent": 100.0},
        "feature_dim": feature_dim,
        "artifacts": {"backbone_only": "backbone_only.pt", "detector_compatible": detector_path.name},
    }
    (OUTPUT_DIR / "transfer_report.json").write_text(json.dumps(report, indent=2))
    return report

def train_one_gpu(config, teacher):
    seed_everything(config.seed)
    train_paths = ssl_paths if config.max_train_images is None else ssl_paths[:config.max_train_images]
    loader = DataLoader(
        SSLImageDataset(train_paths, build_train_transform(config.image_size)),
        batch_size=config.batch_size, shuffle=True, num_workers=config.workers,
        pin_memory=True, persistent_workers=config.workers > 0, drop_last=True,
    )
    assert len(loader) > 0
    encoder, feature_dim = build_student(config)
    model = DistillationStudent(
        encoder, feature_dim, config.teacher_embed_dim, config.global_loss_weight, config.dense_loss_weight
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=config.amp)
    updates_per_epoch = math.ceil(len(loader) / config.grad_accum)
    total_updates = config.epochs * updates_per_epoch
    warmup_updates = config.warmup_epochs * updates_per_epoch
    history, best_loss, global_update = [], float("inf"), 0

    for epoch in range(config.epochs):
        model.train()
        totals = {"loss": 0.0, "global_loss": 0.0, "dense_loss": 0.0}
        started = time.time()
        optimizer.zero_grad(set_to_none=True)
        for step, images in enumerate(loader):
            images = images.to(DEVICE, non_blocking=True)
            with torch.no_grad():
                teacher_global, teacher_dense = teacher(images.float())
            with torch.amp.autocast("cuda", enabled=config.amp):
                loss, pieces = model(images, teacher_global, teacher_dense)
                scaled_loss = loss / config.grad_accum
            assert torch.isfinite(scaled_loss), f"Non-finite loss at epoch {epoch}, step {step}"
            scaler.scale(scaled_loss).backward()
            for name in totals:
                totals[name] += float(loss.detach()) if name == "loss" else float(pieces[name])
            is_update = (step + 1) % config.grad_accum == 0 or (step + 1) == len(loader)
            if is_update:
                lr = set_lr(optimizer, config.learning_rate, config.min_learning_rate, global_update, total_updates, warmup_updates)
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_update += 1
        record = {"epoch": epoch + 1, "images": len(train_paths), "steps": len(loader),
                  "loss": totals["loss"] / len(loader), "global_loss": totals["global_loss"] / len(loader),
                  "dense_loss": totals["dense_loss"] / len(loader), "learning_rate": lr,
                  "seconds": round(time.time() - started, 1)}
        history.append(record)
        state = {
            "epoch": epoch + 1, "best_loss": min(best_loss, record["loss"]), "feature_dim": feature_dim,
            "encoder_state_dict": {k: v.cpu() for k, v in model.encoder.state_dict().items()},
            "global_head_state_dict": {k: v.cpu() for k, v in model.global_head.state_dict().items()},
            "dense_head_state_dict": {k: v.cpu() for k, v in model.dense_head.state_dict().items()},
            "optimizer_state_dict": optimizer.state_dict(), "config": asdict(config),
        }
        torch.save(state, OUTPUT_DIR / "last_ssl.pt")
        if record["loss"] < best_loss:
            best_loss = record["loss"]
            torch.save(state, OUTPUT_DIR / "best_ssl.pt")
        with open(OUTPUT_DIR / "history.csv", "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=history[0].keys())
            writer.writeheader(); writer.writerows(history)
        print(f"Epoch {epoch + 1}/{config.epochs} | loss {record['loss']:.4f} | "
              f"global {record['global_loss']:.4f} | dense {record['dense_loss']:.4f} | {record['seconds']:.0f}s")
    report = export_detector_checkpoint(config, model, feature_dim, history)
    manifest = {"config": asdict(config), "device": torch.cuda.get_device_name(0), "ssl_images": len(train_paths),
                "teacher_frozen": all(not p.requires_grad for p in teacher.parameters()), "history": history,
                "transfer_report": report}
    (OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
    return model.eval(), history, report

print("Standalone DINOv3-to-YOLOv12m components loaded.")

Standalone DINOv3-to-YOLOv12m components loaded.


In [10]:
# Initialize and validate the frozen DINOv3 teacher and graph-routed YOLOv12m student.
seed_everything(cfg.seed)
teacher = load_teacher(cfg)
assert all(not p.requires_grad for p in teacher.parameters()), "Teacher parameters must be frozen"

with torch.no_grad():
    teacher_global, teacher_dense = teacher(torch.randn(2, 3, cfg.image_size, cfg.image_size, device=DEVICE))
assert teacher_global.shape == (2, 768)
assert teacher_dense.shape == (2, 768, 14, 14)

probe_encoder, probe_dim = build_student(cfg)
probe_map = probe_encoder.forward_feature_map(torch.randn(2, 3, cfg.image_size, cfg.image_size, device=DEVICE))
print("Teacher global / dense:", tuple(teacher_global.shape), tuple(teacher_dense.shape))
print("YOLOv12m backbone layers:", probe_encoder.backbone_end, "feature map:", tuple(probe_map.shape), "feature dim:", probe_dim)
assert probe_map.ndim == 4 and probe_dim == probe_map.shape[1]
del probe_encoder, probe_map, teacher_global, teacher_dense
torch.cuda.empty_cache()

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov3_main


Teacher global / dense: (2, 768) (2, 768, 14, 14)
YOLOv12m backbone layers: 9 feature map: (2, 512, 7, 7) feature dim: 512


In [ ]:
# Preview the shared stochastic image transform. Teacher and student receive the same sampled view.
import matplotlib.pyplot as plt

preview_transform = build_train_transform(cfg.image_size)
mean = np.asarray(IMAGENET_MEAN).reshape(3, 1, 1)
std = np.asarray(IMAGENET_STD).reshape(3, 1, 1)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, path in enumerate(ssl_paths[:3]):
    with Image.open(path) as image:
        original = image.convert("RGB")
        axes[row, 0].imshow(original)
    axes[row, 0].set_title("original")
    for column in (1, 2):
        view = preview_transform(original).numpy()
        axes[row, column].imshow(np.transpose(np.clip(view * std + mean, 0, 1), (1, 2, 0)))
        axes[row, column].set_title("shared train view")
    for axis in axes[row]:
        axis.axis("off")
plt.tight_layout()
plt.show()

In [11]:
# Finite-loss and gradient-flow test before the full one-epoch run.
probe_transform = build_train_transform(cfg.image_size)
probe_images = []
for path in ssl_paths[:2]:
    with Image.open(path) as image:
        probe_images.append(probe_transform(image.convert("RGB")))
probe_images = torch.stack(probe_images).to(DEVICE)
encoder, feature_dim = build_student(cfg)
probe_model = DistillationStudent(encoder, feature_dim, cfg.teacher_embed_dim, cfg.global_loss_weight, cfg.dense_loss_weight).to(DEVICE).train()
with torch.no_grad():
    target_global, target_dense = teacher(probe_images.float())
with torch.amp.autocast("cuda", enabled=cfg.amp):
    probe_loss, probe_parts = probe_model(probe_images, target_global, target_dense)
assert torch.isfinite(probe_loss), f"Non-finite loss: {probe_loss.item()}"
probe_loss.backward()
student_grad_norm = sum(p.grad.detach().norm().item() for p in probe_model.encoder.parameters() if p.grad is not None)
teacher_grad_count = sum(p.grad is not None for p in teacher.parameters())
assert student_grad_norm > 0, "No gradients reached the student backbone"
assert teacher_grad_count == 0, "Frozen teacher received gradients"
print({"loss": round(float(probe_loss.detach()), 4), "global": round(float(probe_parts['global_loss']), 4),
       "dense": round(float(probe_parts['dense_loss']), 4), "student_grad_norm": round(student_grad_norm, 4),
       "teacher_grad_tensors": teacher_grad_count})
del encoder, probe_model, probe_images, probe_loss, target_global, target_dense
torch.cuda.empty_cache()

{'loss': 2.0233, 'global': 1.0258, 'dense': 0.9975, 'student_grad_norm': 429.3977, 'teacher_grad_tensors': 0}


/tmp/ipykernel_1135/3029038319.py:20: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print({"loss": round(float(probe_loss), 4), "global": round(float(probe_parts['global_loss']), 4),


In [12]:
# One complete, label-free SSL-pool epoch on the single Colab T4.
trained_model, history, transfer_report = train_one_gpu(cfg, teacher)
print("One-epoch run complete:", history[-1])
print("Transfer coverage:", transfer_report["backbone_tensor_coverage"])
print("Artifacts:", sorted(path.name for path in OUTPUT_DIR.iterdir()))

Epoch 1/1 | loss 1.1669 | global 0.6972 | dense 0.4698 | 393s
One-epoch run complete: {'epoch': 1, 'images': 9180, 'steps': 1147, 'loss': 1.1669372140373888, 'global_loss': 0.6971759563336294, 'dense_loss': 0.469761257963588, 'learning_rate': 0.0003, 'seconds': 392.6}
Transfer coverage: {'matched': 402, 'expected': 402, 'percent': 100.0}
Artifacts: ['backbone_only.pt', 'best_ssl.pt', 'history.csv', 'last_ssl.pt', 'run_manifest.json', 'transfer_report.json', 'yolo12m_dinov3_backbone.pt']


In [13]:
# Safe progress/artifact check for a long Colab execution.
print("CUDA allocated GB:", round(torch.cuda.memory_allocated() / 1e9, 2))
print("Output artifacts:", sorted(path.name for path in OUTPUT_DIR.glob("*")))
history_path = OUTPUT_DIR / "history.csv"
if history_path.exists():
    print(history_path.read_text())

CUDA allocated GB: 0.48
Output artifacts: ['backbone_only.pt', 'best_ssl.pt', 'history.csv', 'last_ssl.pt', 'run_manifest.json', 'transfer_report.json', 'yolo12m_dinov3_backbone.pt']
epoch,images,steps,loss,global_loss,dense_loss,learning_rate,seconds
1,9180,1147,1.1669372140373888,0.6971759563336294,0.469761257963588,0.0003,392.6



In [ ]:
# Image-only representation diagnostics: t-SNE and cosine retrieval. No labels are read or inferred.
from sklearn.manifold import TSNE

@torch.no_grad()
def collect_embeddings(model, paths, image_size, batch_size=16):
    loader = DataLoader(SSLImageDataset(paths, build_eval_transform(image_size)), batch_size=batch_size, shuffle=False, num_workers=2)
    vectors = []
    for images in loader:
        vectors.append(F.normalize(model.encoder(images.to(DEVICE)), dim=1).cpu())
    return torch.cat(vectors).numpy()

diagnostic_paths = ssl_paths[:cfg.diagnostic_images]
embeddings = collect_embeddings(trained_model, diagnostic_paths, cfg.image_size)
perplexity = min(30, max(5, len(diagnostic_paths) // 4))
coordinates = TSNE(n_components=2, perplexity=perplexity, init="pca", learning_rate="auto", random_state=cfg.seed).fit_transform(embeddings)
fig, axis = plt.subplots(figsize=(7, 6))
axis.scatter(coordinates[:, 0], coordinates[:, 1], s=16, alpha=0.75)
axis.set_title("DINOv3-distilled YOLOv12m embeddings (image-only diagnostic)")
axis.set_xlabel("t-SNE 1"); axis.set_ylabel("t-SNE 2")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "embedding_tsne.png", dpi=160); plt.show()

similarities = embeddings @ embeddings.T
np.fill_diagonal(similarities, -np.inf)
query_indices = np.linspace(0, len(diagnostic_paths) - 1, 4, dtype=int)
fig, axes = plt.subplots(len(query_indices), cfg.retrieval_neighbors + 1, figsize=(14, 2.8 * len(query_indices)))
for row, query in enumerate(query_indices):
    neighbours = np.argsort(-similarities[query])[:cfg.retrieval_neighbors]
    display_indices = [query, *neighbours]
    for column, index in enumerate(display_indices):
        with Image.open(diagnostic_paths[index]) as image:
            axes[row, column].imshow(image.convert("RGB"))
        title = "query" if column == 0 else f"cos={similarities[query, index]:.2f}"
        axes[row, column].set_title(title); axes[row, column].axis("off")
fig.suptitle("Cosine nearest neighbours from a fixed SSL-pool sample", y=1.01)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "nearest_neighbours.png", dpi=160, bbox_inches="tight"); plt.show()

In [14]:
# Final artifact verification for the Colab one-epoch validation.
required = {
    "best_ssl.pt", "last_ssl.pt", "history.csv", "run_manifest.json",
    "backbone_only.pt", "yolo12m_dinov3_backbone.pt", "transfer_report.json",
}
produced = {path.name for path in OUTPUT_DIR.iterdir()}
assert required <= produced, f"Missing artifacts: {sorted(required - produced)}"
report = json.loads((OUTPUT_DIR / "transfer_report.json").read_text())
assert report["backbone_tensor_coverage"]["percent"] == 100.0

# Confirm the detector-compatible checkpoint is accepted by Ultralytics.
reloaded = YOLO(str(OUTPUT_DIR / "yolo12m_dinov3_backbone.pt"))
print("Validation passed")
print("Reloaded task:", reloaded.task)
print("History:")
print((OUTPUT_DIR / "history.csv").read_text())
print("Final artifacts:", sorted(produced))

Validation passed
Reloaded task: detect
History:
epoch,images,steps,loss,global_loss,dense_loss,learning_rate,seconds
1,9180,1147,1.1669372140373888,0.6971759563336294,0.469761257963588,0.0003,392.6

Final artifacts: ['backbone_only.pt', 'best_ssl.pt', 'history.csv', 'last_ssl.pt', 'run_manifest.json', 'transfer_report.json', 'yolo12m_dinov3_backbone.pt']
